# Import Libraries

In [1]:
# MODULE 1: LIBRARY IMPORTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("RetailIQ - Environment Ready!")

RetailIQ - Environment Ready!


# Dataset Loading

In [ ]:
# 2.1 DATASET LOADING

df = pd.read_excel("D:online+retail/Online Retail.xlsx")

print("Dataset Loaded Successfully!")

print("Total Records:", df.shape[0])
print("Total Features:", df.shape[1])

display(df.head())

# Dataset Structure

In [ ]:
# 2.2 DATASET STRUCTURE

print("Dataset Information")
print("=" * 50)

print("\nDataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nDetailed Information:")
df.info()

# Statistical Summary

In [ ]:
# 2.3 STATISTICAL ANALYSIS

print("Statistical Summary of Dataset")

display(df.describe(include='all'))

# Missing value Analysis

In [ ]:
# 2.4 MISSING VALUE ANALYSIS

missing_values = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage': 
        (df.isnull().sum() / len(df)) * 100
})

display(missing_values.sort_values(
    by='Missing Percentage',
    ascending=False
))

Visualisation

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    x=missing_values.index,
    y=missing_values['Missing Count']
)

plt.xticks(rotation=45)
plt.title("Missing Values in Each Column")
plt.ylabel("Number of Missing Values")
plt.xlabel("Dataset Features")
plt.show()

Duplicate Analysis

In [ ]:
# 2.5 DUPLICATE ANALYSIS

duplicate_count = df.duplicated().sum()

print("Total Duplicate Records:", duplicate_count)

print(
    "Duplicate Percentage:",
    round((duplicate_count / len(df)) * 100, 2),
    "%"
)

Univariate Analysis

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    df['Quantity'],
    bins=50,
    kde=True
)

plt.title("Distribution of Product Quantity")
plt.xlabel("Quantity")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    df['UnitPrice'],
    bins=50,
    kde=True
)

plt.title("Distribution of Unit Price")
plt.xlabel("Unit Price")
plt.ylabel("Frequency")
plt.show()

Bivariate Analysis

In [ ]:
plt.figure(figsize=(10, 5))

sns.scatterplot(
    data=df.sample(min(5000, len(df)), random_state=42),
    x='Quantity',
    y='UnitPrice',
    alpha=0.5
)

plt.title("Quantity vs Unit Price")
plt.xlabel("Quantity")
plt.ylabel("Unit Price")
plt.show()

Correlation Analysis

In [ ]:
plt.figure(figsize=(8, 5))

correlation = df.select_dtypes(
    include=np.number
).corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
country_counts = df['Country'].value_counts().head(10)

plt.figure(figsize=(12, 5))

sns.barplot(
    x=country_counts.index,
    y=country_counts.values
)

plt.xticks(rotation=45)
plt.title("Top 10 Countries by Transaction Records")
plt.xlabel("Country")
plt.ylabel("Number of Transactions")
plt.show()

# Cleaning and Preprocessing

In [ ]:
# 3.1 CREATING A COPY OF ORIGINAL DATASET

df_clean = df.copy()

print("Original Dataset Shape:", df.shape)
print("Working Dataset Shape:", df_clean.shape)

print("\nWorking dataset created successfully!")

In [ ]:
# 3.2 REMOVING DUPLICATE RECORDS

duplicates_before = df_clean.duplicated().sum()

print("Duplicates Before Removal:", duplicates_before)

df_clean = df_clean.drop_duplicates()

duplicates_after = df_clean.duplicated().sum()

print("Duplicates After Removal:", duplicates_after)
print("Updated Dataset Shape:", df_clean.shape)

In [ ]:
# 3.3 MISSING VALUE ANALYSIS

print("Missing Values Before Treatment:")
display(df_clean.isnull().sum())

In [ ]:
# REMOVING RECORDS WITH MISSING CUSTOMER ID OR DESCRIPTION

df_clean = df_clean.dropna(
    subset=['CustomerID', 'Description']
)

print("Missing Values After Treatment:")
display(df_clean.isnull().sum())

print("\nUpdated Dataset Shape:", df_clean.shape)

In [ ]:
# 3.4 HANDLING INVALID TRANSACTIONS

print("Records Before Filtering:", len(df_clean))

invalid_quantity = (df_clean['Quantity'] <= 0).sum()
invalid_price = (df_clean['UnitPrice'] <= 0).sum()

print("Non-positive Quantities:", invalid_quantity)
print("Non-positive Unit Prices:", invalid_price)

df_clean = df_clean[
    (df_clean['Quantity'] > 0) &
    (df_clean['UnitPrice'] > 0)
].copy()

print("\nRecords After Filtering:", len(df_clean))
print("Updated Dataset Shape:", df_clean.shape)

In [ ]:
# 3.5 DATA TYPE CONVERSION

df_clean['InvoiceDate'] = pd.to_datetime(
    df_clean['InvoiceDate']
)

df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

df_clean['Quantity'] = df_clean['Quantity'].astype(int)

print("Updated Data Types:")
print(df_clean.dtypes)

In [ ]:
# 3.6 NUMERICAL AND CATEGORICAL FEATURES

numerical_features = df_clean.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = df_clean.select_dtypes(
    include='object'
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

In [ ]:
display(df_clean[numerical_features].describe())

In [ ]:
display(df_clean[categorical_features].describe())

# Outlier Detection Using IQR

In [ ]:
# 3.7 OUTLIER DETECTION USING IQR

numeric_columns = ['Quantity', 'UnitPrice']

for col in numeric_columns:

    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[col] < lower_bound) |
        (df_clean[col] > upper_bound)
    ]

    print(f"\nFeature: {col}")
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Outliers Detected:", len(outliers))

In [ ]:
# VISUALIZING OUTLIERS

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(y=df_clean['Quantity'], ax=axes[0])
axes[0].set_title("Quantity Outliers")

sns.boxplot(y=df_clean['UnitPrice'], ax=axes[1])
axes[1].set_title("Unit Price Outliers")

plt.tight_layout()
plt.show()

In [ ]:
# 3.8 MIN-MAX NORMALIZATION

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(
    df_clean[['Quantity', 'UnitPrice']]
)

df_scaled = pd.DataFrame(
    scaled_data,
    columns=['Quantity_Normalized', 'UnitPrice_Normalized'],
    index=df_clean.index
)

display(df_scaled.head())

In [ ]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()

standardized_data = standard_scaler.fit_transform(
    df_clean[['Quantity', 'UnitPrice']]
)

df_standardized = pd.DataFrame(
    standardized_data,
    columns=['Quantity_Standardized', 'UnitPrice_Standardized'],
    index=df_clean.index
)

display(df_standardized.head())

In [ ]:
# 3.9 FINAL DATASET VERIFICATION

print("FINAL CLEANED DATASET")
print("=" * 45)

print("Original Records:", len(df))
print("Cleaned Records:", len(df_clean))

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum().sum())

print("\nRemaining Duplicates:")
print(df_clean.duplicated().sum())

print("\nFinal Dataset Shape:")
print(df_clean.shape)

display(df_clean.head())

# Feature Engineering

In [ ]:
# 4.1 FEATURE ENGINEERING

df_clean['TotalRevenue'] = (
    df_clean['Quantity'] * df_clean['UnitPrice']
)

print("Revenue Feature Created Successfully!")

display(df_clean[
    ['InvoiceNo', 'Quantity', 'UnitPrice', 'TotalRevenue']
].head())

In [ ]:
# 4.2 DATE FEATURE EXTRACTION

df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['Day'] = df_clean['InvoiceDate'].dt.day
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.dayofweek

print("Date Features Created Successfully!")

display(df_clean[
    ['InvoiceDate', 'Year', 'Month', 'Day', 'Hour', 'DayOfWeek']
].head())

In [ ]:
# 4.3 SALES PERFORMANCE

total_revenue = df_clean['TotalRevenue'].sum()

total_transactions = df_clean['InvoiceNo'].nunique()

total_customers = df_clean['CustomerID'].nunique()

total_products = df_clean['StockCode'].nunique()

print("RETAIL BUSINESS PERFORMANCE")
print("=" * 40)

print("Total Revenue: £", round(total_revenue, 2))
print("Total Transactions:", total_transactions)
print("Total Customers:", total_customers)
print("Unique Products:", total_products)

In [ ]:
# 4.4 MONTHLY REVENUE ANALYSIS

monthly_revenue = df_clean.groupby('Month')[
    'TotalRevenue'
].sum()

plt.figure(figsize=(11, 5))

sns.barplot(
    x=monthly_revenue.index,
    y=monthly_revenue.values
)

plt.title("Monthly Revenue Analysis")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.show()

In [ ]:
# 4.5 CUSTOMER SPENDING ANALYSIS

customer_spending = df_clean.groupby('CustomerID').agg(
    TotalSpending=('TotalRevenue', 'sum'),
    TotalQuantity=('Quantity', 'sum'),
    TotalInvoices=('InvoiceNo', 'nunique'),
    AverageOrderValue=('TotalRevenue', 'mean')
).reset_index()

print("CUSTOMER ANALYSIS CREATED!")

display(customer_spending.head())

In [ ]:
# 4.6 CUSTOMER PURCHASE FREQUENCY

customer_frequency = df_clean.groupby(
    'CustomerID'
)['InvoiceNo'].nunique().reset_index()

customer_frequency.columns = [
    'CustomerID',
    'PurchaseFrequency'
]

display(customer_frequency.head())

In [ ]:
# 4.7 RFM ANALYSIS

reference_date = (
    df_clean['InvoiceDate'].max() +
    pd.Timedelta(days=1)
)

rfm = df_clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate',
             lambda x: (reference_date - x.max()).days),

    Frequency=('InvoiceNo', 'nunique'),

    Monetary=('TotalRevenue', 'sum')
).reset_index()

print("RFM ANALYSIS COMPLETED!")

display(rfm.head())

In [ ]:
# 4.8 RFM DISTRIBUTION

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(rfm['Recency'], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Customer Recency")

sns.histplot(rfm['Frequency'], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Customer Frequency")

sns.histplot(rfm['Monetary'], bins=30, kde=True, ax=axes[2])
axes[2].set_title("Customer Monetary Value")

plt.tight_layout()
plt.show()

In [ ]:
# 4.9 RFM SCORING

rfm['R_Score'] = pd.qcut(
    rfm['Recency'].rank(method='first'),
    4,
    labels=[4, 3, 2, 1]
).astype(int)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm['RFM_Score'] = (
    rfm['R_Score'] +
    rfm['F_Score'] +
    rfm['M_Score']
)

display(rfm.head())

In [ ]:
# 4.10 FINAL CUSTOMER FEATURE DATASET

customer_features = rfm.copy()

print("CUSTOMER FEATURE DATASET")
print("=" * 40)

print("Total Customers:", len(customer_features))
print("Total Features:", customer_features.shape[1])

display(customer_features.head())

print("\nStatistical Summary:")
display(customer_features.describe())

Supervised Learning

In [ ]:
# 5.1 REGRESSION DATASET PREPARATION

regression_df = df_clean.groupby('InvoiceNo').agg(
    TotalQuantity=('Quantity', 'sum'),
    AverageUnitPrice=('UnitPrice', 'mean'),
    UniqueProducts=('StockCode', 'nunique'),
    InvoiceRevenue=('TotalRevenue', 'sum'),
    InvoiceDate=('InvoiceDate', 'first')
).reset_index()

# Extract time-based features
regression_df['Month'] = regression_df['InvoiceDate'].dt.month
regression_df['Hour'] = regression_df['InvoiceDate'].dt.hour
regression_df['DayOfWeek'] = regression_df['InvoiceDate'].dt.dayofweek

print("Regression Dataset Created!")
print("Dataset Shape:", regression_df.shape)

display(regression_df.head())

In [ ]:
# 5.2 FEATURE AND TARGET SELECTION

X = regression_df[['TotalQuantity']]

y = regression_df['InvoiceRevenue']

print("Independent Variable:")
display(X.head())

print("\nDependent Variable:")
display(y.head())

In [ ]:
# 5.3 TRAIN-TEST SPLIT

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))

In [ ]:
# 5.4 SIMPLE LINEAR REGRESSION

from sklearn.linear_model import LinearRegression

simple_model = LinearRegression()

simple_model.fit(X_train, y_train)

simple_predictions = simple_model.predict(X_test)

print("Simple Linear Regression Completed!")

print("Intercept:", simple_model.intercept_)
print("Coefficient:", simple_model.coef_[0])

In [ ]:
plt.figure(figsize=(10, 5))

plt.scatter(
    X_test['TotalQuantity'],
    y_test,
    alpha=0.5,
    label='Actual Revenue'
)

plt.scatter(
    X_test['TotalQuantity'],
    simple_predictions,
    alpha=0.5,
    label='Predicted Revenue'
)

plt.xlabel("Total Quantity")
plt.ylabel("Invoice Revenue")
plt.title("Simple Linear Regression")
plt.legend()
plt.show()

In [ ]:
# 5.5 MULTIPLE LINEAR REGRESSION

X_multi = regression_df[
    [
        'TotalQuantity',
        'AverageUnitPrice',
        'UniqueProducts',
        'Month',
        'Hour',
        'DayOfWeek'
    ]
]

y_multi = regression_df['InvoiceRevenue']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi,
    y_multi,
    test_size=0.20,
    random_state=42
)

multiple_model = LinearRegression()

multiple_model.fit(X_train_m, y_train_m)

multiple_predictions = multiple_model.predict(X_test_m)

print("Multiple Linear Regression Completed!")

print("Intercept:", multiple_model.intercept_)

coefficients = pd.DataFrame({
    'Feature': X_multi.columns,
    'Coefficient': multiple_model.coef_
})

display(coefficients)

In [ ]:
# 5.6 POLYNOMIAL REGRESSION

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

polynomial_model = make_pipeline(
    PolynomialFeatures(degree=2),
    LinearRegression()
)

polynomial_model.fit(X_train, y_train)

polynomial_predictions = polynomial_model.predict(X_test)

print("Polynomial Regression Completed!")

In [ ]:
# POLYNOMIAL REGRESSION VISUALIZATION

sorted_indices = np.argsort(X_test['TotalQuantity'].values)

plt.figure(figsize=(10, 5))

plt.scatter(
    X_test['TotalQuantity'],
    y_test,
    alpha=0.4,
    label='Actual Revenue'
)

plt.plot(
    X_test['TotalQuantity'].values[sorted_indices],
    polynomial_predictions[sorted_indices],
    color='red',
    label='Polynomial Prediction'
)

plt.title("Polynomial Regression (Degree 2)")
plt.xlabel("Total Quantity")
plt.ylabel("Invoice Revenue")
plt.legend()
plt.show()

In [ ]:
# 5.8 REGRESSION PERFORMANCE EVALUATION

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def evaluate_regression(actual, predicted):

    mae = mean_absolute_error(actual, predicted)

    mse = mean_squared_error(actual, predicted)

    rmse = np.sqrt(mse)

    r2 = r2_score(actual, predicted)

    return [mae, mse, rmse, r2]


simple_metrics = evaluate_regression(
    y_test, simple_predictions
)

multiple_metrics = evaluate_regression(
    y_test_m, multiple_predictions
)

polynomial_metrics = evaluate_regression(
    y_test, polynomial_predictions
)

print("Regression Evaluation Completed!")

In [ ]:
# 5.9 MODEL COMPARISON

regression_results = pd.DataFrame({
    'Model': [
        'Simple Linear Regression',
        'Multiple Linear Regression',
        'Polynomial Regression'
    ],
    'MAE': [
        simple_metrics[0],
        multiple_metrics[0],
        polynomial_metrics[0]
    ],
    'MSE': [
        simple_metrics[1],
        multiple_metrics[1],
        polynomial_metrics[1]
    ],
    'RMSE': [
        simple_metrics[2],
        multiple_metrics[2],
        polynomial_metrics[2]
    ],
    'R2 Score': [
        simple_metrics[3],
        multiple_metrics[3],
        polynomial_metrics[3]
    ]
})

display(regression_results.round(3))

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=regression_results,
    x='Model',
    y='R2 Score'
)

plt.title("Regression Model Performance Comparison")
plt.xticks(rotation=15)
plt.show()

# Supervised Learning

In [ ]:
# 6.1 CLASSIFICATION DATASET PREPARATION

# Define observation and prediction periods
cutoff_date = (
    df_clean['InvoiceDate'].max() -
    pd.DateOffset(months=3)
)

# Historical transaction data
historical_data = df_clean[
    df_clean['InvoiceDate'] < cutoff_date
].copy()

# Future transaction data
future_data = df_clean[
    df_clean['InvoiceDate'] >= cutoff_date
].copy()

# Historical customer features
classification_df = historical_data.groupby('CustomerID').agg(
    Recency=('InvoiceDate',
             lambda x: (cutoff_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalRevenue', 'sum'),
    AverageQuantity=('Quantity', 'mean'),
    AverageUnitPrice=('UnitPrice', 'mean'),
    UniqueProducts=('StockCode', 'nunique')
).reset_index()

# Identify customers purchasing in the future period
future_customers = future_data['CustomerID'].unique()

# Create binary target
classification_df['RepeatPurchase'] = (
    classification_df['CustomerID'].isin(future_customers)
).astype(int)

print("Classification Dataset Created!")

display(classification_df.head())

print("\nTarget Distribution:")
print(classification_df['RepeatPurchase'].value_counts())

In [ ]:
# 6.2 TARGET VARIABLE DISTRIBUTION

plt.figure(figsize=(7, 5))

sns.countplot(
    data=classification_df,
    x='RepeatPurchase'
)

plt.title("Future Customer Purchase Distribution")
plt.xlabel("Repeat Purchase (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
# 6.3 FEATURE SELECTION

X_class = classification_df[
    [
        'Recency',
        'Frequency',
        'Monetary',
        'AverageQuantity',
        'AverageUnitPrice',
        'UniqueProducts'
    ]
]

y_class = classification_df['RepeatPurchase']

from sklearn.model_selection import train_test_split

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class,
    y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

print("Training Records:", len(X_train_c))
print("Testing Records:", len(X_test_c))

print("\nTraining Target Distribution:")
print(y_train_c.value_counts())

In [ ]:
# 6.4 FEATURE SCALING

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_c)

X_test_scaled = scaler.transform(X_test_c)

print("Feature Scaling Completed!")

In [ ]:
# 6.5 LOGISTIC REGRESSION

from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_model.fit(X_train_scaled, y_train_c)

log_predictions = log_model.predict(X_test_scaled)

print("Logistic Regression Completed!")

In [ ]:
# 6.6 K-NEAREST NEIGHBORS

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train_c)

knn_predictions = knn_model.predict(X_test_scaled)

print("KNN Classification Completed!")

In [ ]:
# 6.7 NAIVE BAYES

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()

nb_model.fit(X_train_scaled, y_train_c)

nb_predictions = nb_model.predict(X_test_scaled)

print("Naive Bayes Classification Completed!")

In [ ]:
# 6.8 DECISION TREE CLASSIFIER

from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train_c, y_train_c)

dt_predictions = dt_model.predict(X_test_c)

print("Decision Tree Classification Completed!")

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(18, 9))

plot_tree(
    dt_model,
    feature_names=X_class.columns,
    class_names=['No Purchase', 'Repeat Purchase'],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree Classification")
plt.show()

In [ ]:
# 6.9 SUPPORT VECTOR MACHINE

from sklearn.svm import SVC

svm_model = SVC(
    kernel='rbf',
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train_c)

svm_predictions = svm_model.predict(X_test_scaled)

print("SVM Classification Completed!")

In [ ]:
# 6.10 CLASSIFICATION EVALUATION

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    classification_report
)

models = {
    'Logistic Regression': (log_model, X_test_scaled, log_predictions),
    'KNN': (knn_model, X_test_scaled, knn_predictions),
    'Naive Bayes': (nb_model, X_test_scaled, nb_predictions),
    'Decision Tree': (dt_model, X_test_c, dt_predictions),
    'SVM': (svm_model, X_test_scaled, svm_predictions)
}

results = []

for name, (model, test_data, predictions) in models.items():

    probabilities = model.predict_proba(test_data)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test_c, predictions),
        'Precision': precision_score(y_test_c, predictions, zero_division=0),
        'Recall': recall_score(y_test_c, predictions, zero_division=0),
        'F1 Score': f1_score(y_test_c, predictions, zero_division=0),
        'AUC': roc_auc_score(y_test_c, probabilities),
        'Log Loss': log_loss(y_test_c, probabilities)
    })

classification_results = pd.DataFrame(results)

display(classification_results.round(4))

In [ ]:
# 6.11 CONFUSION MATRIX

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (name, (_, _, predictions)) in zip(
    axes.flat, models.items()
):

    cm = confusion_matrix(y_test_c, predictions)

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=ax
    )

    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

axes.flat[-1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 6.12 ROC CURVE

from sklearn.metrics import roc_curve

plt.figure(figsize=(10, 6))

for name, (model, test_data, _) in models.items():

    probabilities = model.predict_proba(test_data)[:, 1]

    fpr, tpr, thresholds = roc_curve(
        y_test_c,
        probabilities
    )

    auc_value = roc_auc_score(
        y_test_c,
        probabilities
    )

    plt.plot(
        fpr,
        tpr,
        label=f'{name} (AUC = {auc_value:.3f})'
    )

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

In [ ]:
# 6.13 MODEL PERFORMANCE COMPARISON

plt.figure(figsize=(11, 5))

sns.barplot(
    data=classification_results,
    x='Model',
    y='F1 Score'
)

plt.title("Classification Model Comparison")
plt.xlabel("Machine Learning Algorithm")
plt.ylabel("F1 Score")

plt.xticks(rotation=20)
plt.show()

print("Best Model Based on F1 Score:")

best_model = classification_results.loc[
    classification_results['F1 Score'].idxmax()
]

display(best_model)

In [ ]:
# 7.1 REGRESSION MODEL COMPARISON

print("REGRESSION MODEL PERFORMANCE")
print("=" * 60)

display(regression_results.round(4))

best_regression = regression_results.loc[
    regression_results['R2 Score'].idxmax()
]

print("\nBest Regression Model:")
print(best_regression['Model'])

print("R2 Score:", round(best_regression['R2 Score'], 4))
print("RMSE:", round(best_regression['RMSE'], 4))

In [ ]:
# 7.2 CLASSIFICATION MODEL COMPARISON

print("CLASSIFICATION MODEL PERFORMANCE")
print("=" * 60)

display(classification_results.round(4))

best_classification = classification_results.loc[
    classification_results['F1 Score'].idxmax()
]

print("\nBest Classification Model:")
print(best_classification['Model'])

print("Accuracy:", round(best_classification['Accuracy'], 4))
print("Precision:", round(best_classification['Precision'], 4))
print("Recall:", round(best_classification['Recall'], 4))
print("F1 Score:", round(best_classification['F1 Score'], 4))
print("AUC:", round(best_classification['AUC'], 4))

In [ ]:
# 7.3 CLASSIFICATION PERFORMANCE VISUALIZATION

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC']

classification_results.set_index('Model')[metrics].plot(
    kind='bar',
    figsize=(12, 6)
)

plt.title("Classification Algorithms - Performance Comparison")
plt.xlabel("Machine Learning Models")
plt.ylabel("Performance Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# REGRESSION PERFORMANCE VISUALIZATION

regression_results.set_index('Model')[
    ['MAE', 'RMSE']
].plot(
    kind='bar',
    figsize=(10, 5)
)

plt.title("Regression Models - Error Comparison")
plt.xlabel("Regression Algorithms")
plt.ylabel("Prediction Error")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# 7.4 FINAL PROJECT SUMMARY

print("=" * 60)
print("             RETAILIQ PROJECT SUMMARY")
print("=" * 60)

print("\nDATASET INFORMATION")
print("Original Records:", len(df))
print("Cleaned Records:", len(df_clean))
print("Unique Customers:", df_clean['CustomerID'].nunique())
print("Unique Products:", df_clean['StockCode'].nunique())

print("\nBUSINESS PERFORMANCE")
print("Total Revenue: £", round(
    df_clean['TotalRevenue'].sum(), 2
))

print("Total Transactions:",
      df_clean['InvoiceNo'].nunique())

print("\nREGRESSION RESULTS")
print("Best Model:", best_regression['Model'])
print("R2 Score:", round(best_regression['R2 Score'], 4))

print("\nCLASSIFICATION RESULTS")
print("Best Model:", best_classification['Model'])
print("F1 Score:", round(best_classification['F1 Score'], 4))
print("Accuracy:", round(best_classification['Accuracy'], 4))

print("\nPROJECT STATUS: COMPLETED")
print("=" * 60)

In [ ]:
# 7.8 EXPORTING FINAL PROJECT RESULTS

df_clean.to_csv(
    "RetailIQ_Cleaned_Dataset.csv",
    index=False
)

customer_features.to_csv(
    "RetailIQ_Customer_Features.csv",
    index=False
)

regression_results.to_csv(
    "RetailIQ_Regression_Results.csv",
    index=False
)

classification_results.to_csv(
    "RetailIQ_Classification_Results.csv",
    index=False
)

print("ALL PROJECT OUTPUTS EXPORTED SUCCESSFULLY!")